# Set Covering Problem in Pure Python
## Five Exact Methods for Book Problem 2.15


## Problem Statement

Book problem `2.15` has two binary set-covering models built on the same city map:

1. part `a`: choose fire-station locations to cover all `22` neighborhoods at minimum fixed cost,
2. part `b`: choose at most `4` locations to maximize the population covered according to the printed
   objective in the book.

### Note on part (b)

The printed objective counts a neighborhood once for every selected station that can cover it. That is
the exact model reproduced here. At the end of the notebook we also report, as a classroom note, the
best `unique-population` coverage obtained by four stations.


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import combinations
from math import inf
from time import perf_counter


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


NEIGHBORHOODS = list(range(1, 23))
STATIONS = list(range(1, 25))
COVERAGE = {1: [1, 2],
 2: [1, 19],
 3: [2, 3, 19],
 4: [19, 20],
 5: [10, 17, 19],
 6: [18, 19],
 7: [17, 18, 22],
 8: [17, 22],
 9: [15, 16, 17],
 10: [10, 11, 16],
 11: [13, 15, 16],
 12: [14, 15, 21],
 13: [14, 21],
 14: [12, 13],
 15: [7, 13, 14, 21],
 16: [7, 21],
 17: [8, 9, 10, 11],
 18: [3, 4, 8, 9],
 19: [2, 3, 4],
 20: [4, 5],
 21: [7, 8],
 22: [5, 6, 7],
 23: [5, 6],
 24: [2, 4, 6]}
FIXED_COST = {1: 529,
 2: 678,
 3: 816,
 4: 866,
 5: 370,
 6: 357,
 7: 788,
 8: 767,
 9: 640,
 10: 340,
 11: 575,
 12: 387,
 13: 565,
 14: 503,
 15: 731,
 16: 759,
 17: 827,
 18: 555,
 19: 412,
 20: 600,
 21: 810,
 22: 568,
 23: 776,
 24: 798}
POPULATION = {1: 65333,
 2: 103022,
 3: 44088,
 4: 56933,
 5: 100358,
 6: 166906,
 7: 78097,
 8: 96991,
 9: 47830,
 10: 103087,
 11: 98172,
 12: 67638,
 13: 169659,
 14: 151544,
 15: 126496,
 16: 94383,
 17: 103975,
 18: 100276,
 19: 98257,
 20: 65440,
 21: 92170,
 22: 8971}

ALL_MASK = (1 << len(NEIGHBORHOODS)) - 1
NEIGHBORHOOD_BIT = {neighborhood: 1 << (neighborhood - 1) for neighborhood in NEIGHBORHOODS}
STATION_MASK = {station: sum(NEIGHBORHOOD_BIT[neighborhood] for neighborhood in COVERAGE[station]) for station in STATIONS}
PART_B_WEIGHT = {station: sum(POPULATION[neighborhood] for neighborhood in COVERAGE[station]) for station in STATIONS}

DOMINATED = []
for station in STATIONS:
    covered = set(COVERAGE[station])
    for other in STATIONS:
        if station == other:
            continue
        if covered <= set(COVERAGE[other]) and FIXED_COST[other] <= FIXED_COST[station]:
            DOMINATED.append(station)
            break
DOMINATED = sorted(set(DOMINATED))
PART_A_STATIONS = [station for station in STATIONS if station not in DOMINATED]

MANDATORY_STATIONS = []
for neighborhood in NEIGHBORHOODS:
    candidates = [station for station in PART_A_STATIONS if neighborhood in COVERAGE[station]]
    if len(candidates) == 1:
        MANDATORY_STATIONS.append(candidates[0])
MANDATORY_STATIONS = sorted(set(MANDATORY_STATIONS))
MANDATORY_MASK = 0
MANDATORY_COST = 0
for station in MANDATORY_STATIONS:
    MANDATORY_MASK |= STATION_MASK[station]
    MANDATORY_COST += FIXED_COST[station]

EXPECTED_PART_A = {"stations": [1, 4, 7, 10, 12, 14, 18, 22], "total_cost": 4536}
EXPECTED_PART_B = {"stations": [11, 12, 15, 17], "objective": 1598298}
EXPECTED_PART_B_UNIQUE = {"stations": [9, 15, 17, 24], "unique_population": 1489265}


def choose_cover_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (-left["total_cost"], tuple(-station for station in left["stations"]))
    right_key = (-right["total_cost"], tuple(-station for station in right["stations"]))
    return right if right_key > left_key else left


def choose_population_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["objective"], tuple(left["stations"]))
    right_key = (right["objective"], tuple(right["stations"]))
    return right if right_key > left_key else left


def build_cover_solution(stations):
    stations = sorted(set(stations))
    covered_mask = 0
    total_cost = 0
    for station in stations:
        covered_mask |= STATION_MASK[station]
        total_cost += FIXED_COST[station]
    if covered_mask != ALL_MASK:
        return None
    return {"stations": stations, "total_cost": total_cost}


def build_population_solution(stations):
    stations = sorted(set(stations))
    if len(stations) > 4:
        return None
    objective = sum(PART_B_WEIGHT[station] for station in stations)
    covered = set()
    for station in stations:
        covered.update(COVERAGE[station])
    return {"stations": stations, "objective": objective, "unique_population": sum(POPULATION[neighborhood] for neighborhood in covered)}


@timed
def solve_cover_naive():
    best = None
    candidate_stations = PART_A_STATIONS
    station_count = len(candidate_stations)
    for mask in range(1 << station_count):
        chosen = [candidate_stations[index] for index in range(station_count) if (mask >> index) & 1]
        candidate = build_cover_solution(chosen)
        best = choose_cover_better(best, candidate)
    return best


@timed
def solve_cover_backtracking():
    best = None
    ordered_stations = sorted(PART_A_STATIONS, key=lambda station: FIXED_COST[station])
    suffix_union = [0] * (len(ordered_stations) + 1)
    for index in range(len(ordered_stations) - 1, -1, -1):
        suffix_union[index] = suffix_union[index + 1] | STATION_MASK[ordered_stations[index]]

    def backtrack(index, chosen, covered_mask, current_cost):
        nonlocal best
        if best is not None and current_cost >= best["total_cost"]:
            return
        if covered_mask == ALL_MASK:
            best = choose_cover_better(best, {"stations": sorted(chosen), "total_cost": current_cost})
            return
        if index == len(ordered_stations):
            return
        if (covered_mask | suffix_union[index]) != ALL_MASK:
            return
        station = ordered_stations[index]
        backtrack(index + 1, chosen + [station], covered_mask | STATION_MASK[station], current_cost + FIXED_COST[station])
        backtrack(index + 1, chosen, covered_mask, current_cost)

    backtrack(0, [], 0, 0)
    return best


@timed
def solve_cover_reduced_search():
    remaining = [station for station in PART_A_STATIONS if station not in MANDATORY_STATIONS]
    best = None
    suffix_union = [0] * (len(remaining) + 1)
    for index in range(len(remaining) - 1, -1, -1):
        suffix_union[index] = suffix_union[index + 1] | STATION_MASK[remaining[index]]

    def backtrack(index, chosen, covered_mask, current_cost):
        nonlocal best
        if best is not None and current_cost >= best["total_cost"]:
            return
        if covered_mask == ALL_MASK:
            best = choose_cover_better(best, {"stations": sorted(chosen), "total_cost": current_cost})
            return
        if index == len(remaining):
            return
        if (covered_mask | suffix_union[index]) != ALL_MASK:
            return
        station = remaining[index]
        backtrack(index + 1, chosen + [station], covered_mask | STATION_MASK[station], current_cost + FIXED_COST[station])
        backtrack(index + 1, chosen, covered_mask, current_cost)

    backtrack(0, list(MANDATORY_STATIONS), MANDATORY_MASK, MANDATORY_COST)
    return best


@timed
def solve_cover_dynamic_programming():
    layers = [{0: 0}]
    parents = []
    for station in PART_A_STATIONS:
        previous = layers[-1]
        current = dict(previous)
        parent = {}
        for covered_mask, current_cost in previous.items():
            new_mask = covered_mask | STATION_MASK[station]
            new_cost = current_cost + FIXED_COST[station]
            if new_cost < current.get(new_mask, inf):
                current[new_mask] = new_cost
                parent[new_mask] = (covered_mask, station)
        layers.append(current)
        parents.append(parent)
    mask = ALL_MASK
    chosen = []
    for index in range(len(PART_A_STATIONS) - 1, -1, -1):
        if mask in parents[index]:
            previous_mask, station = parents[index][mask]
            chosen.append(station)
            mask = previous_mask
    return build_cover_solution(chosen)


@timed
def solve_cover_branch_and_bound():
    best = solve_cover_reduced_search()[0]
    candidate_stations = tuple(PART_A_STATIONS)
    minimum_cost = min(FIXED_COST[station] for station in candidate_stations)

    def branch(chosen, covered_mask, current_cost, available):
        nonlocal best
        if current_cost >= best["total_cost"]:
            return
        if covered_mask == ALL_MASK:
            best = choose_cover_better(best, {"stations": sorted(chosen), "total_cost": current_cost})
            return
        available_union = covered_mask
        for station in available:
            available_union |= STATION_MASK[station]
        if available_union != ALL_MASK:
            return
        uncovered = [neighborhood for neighborhood in NEIGHBORHOODS if not (covered_mask & NEIGHBORHOOD_BIT[neighborhood])]
        max_new = max(((STATION_MASK[station] & (ALL_MASK ^ covered_mask)).bit_count() for station in available), default=0)
        if max_new == 0:
            return
        lower_bound = current_cost + ((len(uncovered) + max_new - 1) // max_new) * minimum_cost
        if lower_bound >= best["total_cost"]:
            return
        target = min(
            uncovered,
            key=lambda neighborhood: sum(1 for station in available if neighborhood in COVERAGE[station]),
        )
        options = [station for station in available if target in COVERAGE[station]]
        options.sort(key=lambda station: (FIXED_COST[station], -((STATION_MASK[station] & (ALL_MASK ^ covered_mask)).bit_count())))
        for station in options:
            next_available = tuple(candidate for candidate in available if candidate != station)
            branch(chosen + [station], covered_mask | STATION_MASK[station], current_cost + FIXED_COST[station], next_available)

    branch([], 0, 0, candidate_stations)
    return best


@timed
def solve_population_naive():
    best = None
    for size in range(5):
        for chosen in combinations(STATIONS, size):
            best = choose_population_better(best, build_population_solution(chosen))
    return best


@timed
def solve_population_backtracking():
    best = None
    ordered = sorted(STATIONS, key=lambda station: PART_B_WEIGHT[station], reverse=True)

    def branch(index, chosen, current_score):
        nonlocal best
        remaining_slots = 4 - len(chosen)
        optimistic = current_score + sum(PART_B_WEIGHT[station] for station in ordered[index : index + remaining_slots])
        if best is not None and optimistic <= best["objective"]:
            return
        if len(chosen) == 4 or index == len(ordered):
            best = choose_population_better(best, build_population_solution(chosen))
            return
        station = ordered[index]
        branch(index + 1, chosen + [station], current_score + PART_B_WEIGHT[station])
        branch(index + 1, chosen, current_score)

    branch(0, [], 0)
    return best


@timed
def solve_population_reduced_search():
    chosen = sorted(STATIONS, key=lambda station: PART_B_WEIGHT[station], reverse=True)[:4]
    return build_population_solution(chosen)


@timed
def solve_population_dynamic_programming():
    dp = [[None] * 5 for _ in range(len(STATIONS) + 1)]
    dp[0][0] = []
    for index, station in enumerate(STATIONS, start=1):
        for used in range(5):
            if dp[index - 1][used] is not None and dp[index][used] is None:
                dp[index][used] = list(dp[index - 1][used])
            if used > 0 and dp[index - 1][used - 1] is not None:
                candidate = dp[index - 1][used - 1] + [station]
                candidate_solution = build_population_solution(candidate)
                existing_solution = build_population_solution(dp[index][used]) if dp[index][used] is not None else None
                if existing_solution is None or choose_population_better(existing_solution, candidate_solution) is candidate_solution:
                    dp[index][used] = candidate
    best = None
    for used in range(5):
        if dp[len(STATIONS)][used] is not None:
            best = choose_population_better(best, build_population_solution(dp[len(STATIONS)][used]))
    return best


@timed
def solve_population_branch_and_bound():
    ordered = sorted(STATIONS, key=lambda station: PART_B_WEIGHT[station], reverse=True)
    best = build_population_solution(ordered[:4])

    def branch(index, chosen, current_score):
        nonlocal best
        remaining_slots = 4 - len(chosen)
        optimistic = current_score + sum(PART_B_WEIGHT[station] for station in ordered[index : index + remaining_slots])
        if optimistic <= best["objective"]:
            return
        if len(chosen) == 4 or index == len(ordered):
            best = choose_population_better(best, build_population_solution(chosen))
            return
        station = ordered[index]
        branch(index + 1, chosen + [station], current_score + PART_B_WEIGHT[station])
        branch(index + 1, chosen, current_score)

    branch(0, [], 0)
    return best


def solve_unique_population_reference():
    best = None
    for chosen in combinations(STATIONS, 4):
        covered = set()
        for station in chosen:
            covered.update(COVERAGE[station])
        candidate = {"stations": list(chosen), "unique_population": sum(POPULATION[neighborhood] for neighborhood in covered)}
        if best is None or (candidate["unique_population"], tuple(candidate["stations"])) > (best["unique_population"], tuple(best["stations"])):
            best = candidate
    return best


## 1. Naive Enumeration


In [2]:
cover_naive_result, cover_naive_time = solve_cover_naive()
population_naive_result, population_naive_time = solve_population_naive()
print("Part (a) naive result:", cover_naive_result)
print("Part (b) naive result:", population_naive_result)
print(f"Times -> part_a={cover_naive_time:.6f}s part_b={population_naive_time:.6f}s")
assert cover_naive_result == EXPECTED_PART_A
assert population_naive_result["stations"] == EXPECTED_PART_B["stations"]
assert population_naive_result["objective"] == EXPECTED_PART_B["objective"]


Part (a) naive result: {'stations': [1, 4, 7, 10, 12, 14, 18, 22], 'total_cost': 4536}
Part (b) naive result: {'stations': [11, 12, 15, 17], 'objective': 1598298, 'unique_population': 1058429}
Times -> part_a=4.400726s part_b=0.018256s


## 2. Backtracking With Pruning


In [3]:
cover_backtracking_result, cover_backtracking_time = solve_cover_backtracking()
population_backtracking_result, population_backtracking_time = solve_population_backtracking()
print("Part (a) backtracking result:", cover_backtracking_result)
print("Part (b) backtracking result:", population_backtracking_result)
print(f"Times -> part_a={cover_backtracking_time:.6f}s part_b={population_backtracking_time:.6f}s")
assert cover_backtracking_result == EXPECTED_PART_A
assert population_backtracking_result["stations"] == EXPECTED_PART_B["stations"]
assert population_backtracking_result["objective"] == EXPECTED_PART_B["objective"]


Part (a) backtracking result: {'stations': [1, 4, 7, 10, 12, 14, 18, 22], 'total_cost': 4536}
Part (b) backtracking result: {'stations': [11, 12, 15, 17], 'objective': 1598298, 'unique_population': 1058429}
Times -> part_a=0.003745s part_b=0.000014s


## 3. Constraint-Driven Reduced Search


In [4]:
print("Dominated stations removed exactly for part (a):", DOMINATED)
print("Mandatory stations after the exact presolve:", MANDATORY_STATIONS)
cover_reduced_result, cover_reduced_time = solve_cover_reduced_search()
population_reduced_result, population_reduced_time = solve_population_reduced_search()
print("Part (a) reduced-search result:", cover_reduced_result)
print("Part (b) reduced-search result:", population_reduced_result)
print(f"Times -> part_a={cover_reduced_time:.6f}s part_b={population_reduced_time:.6f}s")
assert cover_reduced_result == EXPECTED_PART_A
assert population_reduced_result["stations"] == EXPECTED_PART_B["stations"]
assert population_reduced_result["objective"] == EXPECTED_PART_B["objective"]


Dominated stations removed exactly for part (a): [13, 16, 23]
Mandatory stations after the exact presolve: [4, 14]
Part (a) reduced-search result: {'stations': [1, 4, 7, 10, 12, 14, 18, 22], 'total_cost': 4536}
Part (b) reduced-search result: {'stations': [11, 12, 15, 17], 'objective': 1598298, 'unique_population': 1058429}
Times -> part_a=0.000852s part_b=0.000007s


## 4. Dynamic Programming


In [5]:
cover_dp_result, cover_dp_time = solve_cover_dynamic_programming()
population_dp_result, population_dp_time = solve_population_dynamic_programming()
print("Part (a) dynamic-programming result:", cover_dp_result)
print("Part (b) dynamic-programming result:", population_dp_result)
print(f"Times -> part_a={cover_dp_time:.6f}s part_b={population_dp_time:.6f}s")
assert cover_dp_result == EXPECTED_PART_A
assert population_dp_result["stations"] == EXPECTED_PART_B["stations"]
assert population_dp_result["objective"] == EXPECTED_PART_B["objective"]


Part (a) dynamic-programming result: {'stations': [1, 4, 7, 10, 12, 14, 18, 22], 'total_cost': 4536}
Part (b) dynamic-programming result: {'stations': [11, 12, 15, 17], 'objective': 1598298, 'unique_population': 1058429}
Times -> part_a=0.016122s part_b=0.000250s


## 5. Branch And Bound


In [6]:
cover_bnb_result, cover_bnb_time = solve_cover_branch_and_bound()
population_bnb_result, population_bnb_time = solve_population_branch_and_bound()
print("Part (a) branch-and-bound result:", cover_bnb_result)
print("Part (b) branch-and-bound result:", population_bnb_result)
print(f"Times -> part_a={cover_bnb_time:.6f}s part_b={population_bnb_time:.6f}s")
assert cover_bnb_result == EXPECTED_PART_A
assert population_bnb_result["stations"] == EXPECTED_PART_B["stations"]
assert population_bnb_result["objective"] == EXPECTED_PART_B["objective"]


AttributeError: 'int' object has no attribute 'bit_count'

## Summary


In [ ]:
part_a_rows = [
    ("Naive enumeration", cover_naive_result, cover_naive_time),
    ("Backtracking with pruning", cover_backtracking_result, cover_backtracking_time),
    ("Constraint-driven reduced search", cover_reduced_result, cover_reduced_time),
    ("Dynamic programming", cover_dp_result, cover_dp_time),
    ("Branch and Bound", cover_bnb_result, cover_bnb_time),
]

part_b_rows = [
    ("Naive enumeration", population_naive_result, population_naive_time),
    ("Backtracking with pruning", population_backtracking_result, population_backtracking_time),
    ("Constraint-driven reduced search", population_reduced_result, population_reduced_time),
    ("Dynamic programming", population_dp_result, population_dp_time),
    ("Branch and Bound", population_bnb_result, population_bnb_time),
]

print("Part (a): minimum-cost covering plan")
for method_name, result, elapsed in part_a_rows:
    print(f"{method_name:35s} cost={result['total_cost']:4d} stations={result['stations']} time={elapsed:.6f}s")

print("\nPart (b): printed book objective")
for method_name, result, elapsed in part_b_rows:
    print(
        f"{method_name:35s} objective={result['objective']:7d} "
        f"stations={result['stations']} unique_population={result['unique_population']:7d} time={elapsed:.6f}s"
    )

reference_unique = solve_unique_population_reference()
print("\nClassroom note: best unique-population coverage with four stations:", reference_unique)
assert reference_unique == EXPECTED_PART_B_UNIQUE
